In [ ]:
import numpy as np
import pandas as pd
import time

from dataclasses import dataclass
from typing import Tuple, List, Dict, Optional

from sklearn.model_selection import train_test_split
from sklearn.svm import LinearSVC

In [ ]:
@dataclass
class BMOConfig:
    population_size: int = 20
    max_generations: int = 100

    alpha: float = 0.9
    beta: float = 0.1

    threshold: float = 0.5
    min_features: int = 1

    val_size: float = 0.2
    random_state: int = 42

    # penis length trong BMO
    pl: int = 6

In [ ]:
def continuous_to_mask(
    vector: np.ndarray,
    threshold: float,
    min_features: int,
    rng: np.random.Generator
) -> np.ndarray:
    mask = (vector > threshold).astype(int)

    if mask.sum() < min_features:
        idx = rng.choice(len(mask), size=min_features, replace=False)
        mask[idx] = 1

    return mask

def evaluate_feature_subset_bmo(
    X_train_all: pd.DataFrame,
    X_val_all: pd.DataFrame,
    y_train: np.ndarray,
    y_val: np.ndarray,
    mask: np.ndarray,
    config: BMOConfig
) -> Tuple[float, float, int]:
    """
    Fitness y chang ý tưởng bạn cung cấp:

    fitness = alpha * (1 - accuracy) + beta * feature_ratio

    Fitness càng nhỏ càng tốt.
    """

    if np.sum(mask) == 0:
        return 1.0, 0.0, 0

    selected_cols = X_train_all.columns[mask == 1]

    X_tr = X_train_all[selected_cols]
    X_val = X_val_all[selected_cols]

    if X_tr.shape[1] == 0:
        return 1.0, 0.0, 0

    model = LinearSVC(C=1.0, max_iter=2000, random_state=config.random_state)
    model.fit(X_tr, y_train)

    accuracy = model.score(X_val, y_val)

    feature_ratio = np.sum(mask) / len(mask)

    fitness = config.alpha * (1 - accuracy) + config.beta * feature_ratio

    return fitness, accuracy, int(np.sum(mask))

In [ ]:
class BMOFeatureSelector:
    def __init__(self, config: Optional[BMOConfig] = None):
        self.config = config if config is not None else BMOConfig()
        self.rng = np.random.default_rng(self.config.random_state)

        self.best_mask_: Optional[np.ndarray] = None
        self.best_fitness_: float = np.inf
        self.best_accuracy_: float = 0.0
        self.best_num_features_: int = 0

        self.history_: List[Dict] = []
        self.feature_selection_time_: float = 0.0

    def _initialize_population(self, n_features: int) -> np.ndarray:
        """
        Khởi tạo population continuous trong [0, 1].
        Mỗi cá thể là một vector feature.
        """
        population = self.rng.uniform(
            low=0.0,
            high=1.0,
            size=(self.config.population_size, n_features)
        )

        return population

    def fit(self, X: pd.DataFrame, y: np.ndarray):
        fs_start = time.perf_counter()

        X_train_all, X_val_all, y_train, y_val = train_test_split(
            X,
            y,
            test_size=self.config.val_size,
            random_state=self.config.random_state,
            stratify=y
        )

        n_features = X.shape[1]

        population = self._initialize_population(n_features)

        masks = []
        fitnesses = np.zeros(self.config.population_size)
        accuracies = np.zeros(self.config.population_size)
        num_features = np.zeros(self.config.population_size, dtype=int)

        # =========================
        # 1. ĐÁNH GIÁ QUẦN THỂ BAN ĐẦU
        # =========================
        for i in range(self.config.population_size):
            mask = continuous_to_mask(
                population[i],
                self.config.threshold,
                self.config.min_features,
                self.rng
            )

            fit_i, acc_i, nf_i = evaluate_feature_subset_bmo(
                X_train_all,
                X_val_all,
                y_train,
                y_val,
                mask,
                self.config
            )

            masks.append(mask)
            fitnesses[i] = fit_i
            accuracies[i] = acc_i
            num_features[i] = nf_i

            if fit_i < self.best_fitness_:
                self.best_fitness_ = fit_i
                self.best_mask_ = mask.copy()
                self.best_accuracy_ = acc_i
                self.best_num_features_ = nf_i

        masks = np.array(masks)

        # Sắp xếp population theo fitness tăng dần
        sorted_idx = np.argsort(fitnesses)
        population = population[sorted_idx]
        masks = masks[sorted_idx]
        fitnesses = fitnesses[sorted_idx]
        accuracies = accuracies[sorted_idx]
        num_features = num_features[sorted_idx]

        # =========================
        # 2. VÒNG LẶP BMO
        # =========================
        for gen in range(self.config.max_generations):
            gen_start = time.perf_counter()

            new_population = []

            for i in range(self.config.population_size):
                dad_idx = self.rng.integers(0, self.config.population_size)
                mom_idx = self.rng.integers(0, self.config.population_size)

                dad = population[dad_idx]
                mom = population[mom_idx]

                # Tránh self-mating
                if dad_idx == mom_idx:
                    offspring = population[i].copy()
                else:
                    # Nếu khoảng cách <= pl thì mating bình thường
                    if abs(dad_idx - mom_idx) <= self.config.pl:
                        p = self.rng.random()
                        q = 1.0 - p

                        offspring = p * dad + q * mom

                    # Ngược lại sperm casting để exploration
                    else:
                        rand = self.rng.random()
                        offspring = rand * mom

                offspring = np.clip(offspring, 0.0, 1.0)
                new_population.append(offspring)

            new_population = np.array(new_population)

            new_masks = []
            new_fitnesses = np.zeros(self.config.population_size)
            new_accuracies = np.zeros(self.config.population_size)
            new_num_features = np.zeros(self.config.population_size, dtype=int)

            # =========================
            # 3. ĐÁNH GIÁ OFFSPRING
            # =========================
            for i in range(self.config.population_size):
                mask = continuous_to_mask(
                    new_population[i],
                    self.config.threshold,
                    self.config.min_features,
                    self.rng
                )

                fit_i, acc_i, nf_i = evaluate_feature_subset_bmo(
                    X_train_all,
                    X_val_all,
                    y_train,
                    y_val,
                    mask,
                    self.config
                )

                new_masks.append(mask)
                new_fitnesses[i] = fit_i
                new_accuracies[i] = acc_i
                new_num_features[i] = nf_i

                if fit_i < self.best_fitness_:
                    self.best_fitness_ = fit_i
                    self.best_mask_ = mask.copy()
                    self.best_accuracy_ = acc_i
                    self.best_num_features_ = nf_i

            new_masks = np.array(new_masks)

            # =========================
            # 4. GỘP CHA MẸ + CON
            # =========================
            combined_population = np.vstack([population, new_population])
            combined_masks = np.vstack([masks, new_masks])
            combined_fitnesses = np.concatenate([fitnesses, new_fitnesses])
            combined_accuracies = np.concatenate([accuracies, new_accuracies])
            combined_num_features = np.concatenate([num_features, new_num_features])

            # Chọn lại top population_size cá thể tốt nhất
            sorted_idx = np.argsort(combined_fitnesses)
            selected_idx = sorted_idx[:self.config.population_size]

            population = combined_population[selected_idx]
            masks = combined_masks[selected_idx]
            fitnesses = combined_fitnesses[selected_idx]
            accuracies = combined_accuracies[selected_idx]
            num_features = combined_num_features[selected_idx]

            gen_time = time.perf_counter() - gen_start

            self.history_.append({
                "generation": gen + 1,
                "best_fitness": float(fitnesses[0]),
                "mean_fitness": float(np.mean(fitnesses)),
                "best_accuracy": float(accuracies[0]),
                "mean_accuracy": float(np.mean(accuracies)),
                "best_num_features": int(num_features[0]),
                "mean_num_features": float(np.mean(num_features)),
                "gen_time_sec": gen_time
            })

            print(
                f"Generation {gen + 1:03d} | "
                f"Best fitness = {fitnesses[0]:.6f} | "
                f"Best accuracy = {accuracies[0]:.6f} | "
                f"Selected features = {num_features[0]} | "
                f"Time = {gen_time:.4f}s"
            )

        self.feature_selection_time_ = time.perf_counter() - fs_start

        print(f"\n[Timing] Total feature selection time: {self.feature_selection_time_:.4f} seconds")

        return self

    def get_selected_features(self, feature_names: List[str]) -> List[str]:
        if self.best_mask_ is None:
            raise ValueError("Model chưa fit.")
        return [f for f, m in zip(feature_names, self.best_mask_) if m == 1]

    def transform(self, X: pd.DataFrame) -> pd.DataFrame:
        if self.best_mask_ is None:
            raise ValueError("Model chưa fit.")

        selected_cols = X.columns[self.best_mask_ == 1]
        return X[selected_cols]

    def fit_transform(self, X: pd.DataFrame, y: np.ndarray) -> pd.DataFrame:
        self.fit(X, y)
        return self.transform(X)

In [ ]:
if __name__ == "__main__":
    X_path = "/kaggle/input/datasets/lfreedom2750/data-time-complexity/X_train.csv"
    y_path = "/kaggle/input/datasets/lfreedom2750/data-time-complexity/y_train.csv"

    X = pd.read_csv(X_path)
    y = pd.read_csv(y_path).values.ravel()

    config = BMOConfig(
        population_size=20,
        max_generations=100,
        alpha=0.9,
        beta=0.1,
        threshold=0.5,
        min_features=1,
        val_size=0.2,
        random_state=42,
        pl=6
    )

    selector = BMOFeatureSelector(config=config)
    selector.fit(X, y)

    selected_features = selector.get_selected_features(list(X.columns))
    X_selected = selector.transform(X)

    print("\n========== KẾT QUẢ CUỐI ==========")
    print(f"Best fitness       : {selector.best_fitness_:.6f}")
    print(f"Best accuracy      : {selector.best_accuracy_:.6f}")
    print(f"Số feature đã chọn : {selector.best_num_features_}")
    print(f"Tên feature chọn   : {selected_features}")

    X_selected.to_csv("X_train_selected_bmo.csv", index=False)

    pd.DataFrame({"selected_feature": selected_features}).to_csv(
        "selected_features_bmo.csv",
        index=False
    )

    pd.DataFrame(selector.history_).to_csv(
        "bmo_history.csv",
        index=False
    )

    print("\nĐã lưu:")
    print("- X_train_selected_bmo.csv")
    print("- selected_features_bmo.csv")
    print("- bmo_history.csv")

In [1]:
import pandas as pd

df = pd.read_csv('/kaggle/input/datasets/lfreedom2750/selected-features/selected_features_bmo.csv')

selected_features = df.iloc[:, 0].values  # lấy cột đầu tiên

print(selected_features)

['AAK1' 'ABCF1' 'AC090498.1' 'ACAP1' 'ACP5' 'ADAM15' 'ADPGK' 'AGR3'
 'AHCYL1' 'AHR' 'ALCAM' 'ALDOA' 'AMD1' 'AP1S1' 'AP2S1' 'AP3D1' 'APOC1'
 'APOL6' 'APP' 'ARF3' 'ARF4' 'ARF6' 'ARL2' 'ARPC1A' 'ARPC5' 'ARPP19'
 'ASPH' 'ATF3' 'ATP1B1' 'ATP5H' 'ATP6AP1' 'AUP1' 'B2M' 'B4GALT1' 'B4GALT5'
 'BLOC1S1' 'BLVRB' 'BSG' 'BST2' 'BTG2' 'BUD31' 'C11orf31' 'C1QA' 'C1QC'
 'C1orf122' 'C2' 'C4orf48' 'C5AR1' 'C7orf73' 'CALM2' 'CAMTA1' 'CAPG'
 'CCL18' 'CD14' 'CD151' 'CD27' 'CD302' 'CD37' 'CD3E' 'CD47' 'CD48' 'CD59'
 'CD68' 'CD69' 'CD74' 'CD79B' 'CD81' 'CDIPT' 'CEBPB' 'CHD9' 'CHMP2B'
 'CHMP4B' 'CLDN3' 'CLDN4' 'CLTA' 'CLTC' 'CMPK1' 'CNPY3' 'COA3' 'COMT'
 'COPA' 'COPE' 'CORO1A' 'CPM' 'CPVL' 'CSF1R' 'CSNK2B' 'CTNNA1' 'CTNNB1'
 'CTSA' 'CTSB' 'CTSH' 'CWC15' 'CXCL16' 'CXCL17' 'CXCL2' 'CYB5A' 'CYBRD1'
 'CYTIP' 'DAPK1' 'DCXR' 'DDRGK1' 'DDX18' 'DDX3Y' 'DUSP23' 'DUSP6'
 'DYNC1I2' 'DYNLL1' 'EEF1B2' 'EEF1D' 'EEF2' 'EIF1AX' 'EIF1AY' 'EIF3D'
 'EIF3J' 'EIF3L' 'EIF4A2' 'EIF4A3' 'EIF4EBP1' 'EIF4G1' 'EIF5' 'EIF5A'
 'EIF6' 'ELF

In [2]:
import pandas as pd

# Đọc dữ liệu
X_train = pd.read_csv("/kaggle/input/datasets/lfreedom2750/data-time-complexity/X_train.csv")
X_val = pd.read_csv("/kaggle/input/datasets/lfreedom2750/data-time-complexity/X_val.csv")
X_test = pd.read_csv("/kaggle/input/datasets/lfreedom2750/data-time-complexity/X_test.csv")

y_train = pd.read_csv("/kaggle/input/datasets/lfreedom2750/data-time-complexity/y_train.csv").iloc[:, 0].values
y_val = pd.read_csv("/kaggle/input/datasets/lfreedom2750/data-time-complexity/y_val.csv").iloc[:, 0].values
y_test = pd.read_csv("/kaggle/input/datasets/lfreedom2750/data-time-complexity/y_test.csv").iloc[:, 0].values

# Áp feature selected lên cả 3 tập
X_train_selected = X_train[selected_features].copy()
X_val_selected = X_val[selected_features].copy()
X_test_selected = X_test[selected_features].copy()

print("Train selected shape:", X_train_selected.shape)
print("Val selected shape:", X_val_selected.shape)
print("Test selected shape:", X_test_selected.shape)

Train selected shape: (16000, 479)
Val selected shape: (2000, 479)
Test selected shape: (2000, 479)


In [3]:
from sklearn.metrics import classification_report, accuracy_score, f1_score, precision_score, recall_score

def compute_metrics(y_true, y_pred, model_name):
    report = classification_report(y_true, y_pred,
                                   target_names=CLASS_NAMES,
                                   output_dict=True, zero_division=0)
    row = {"model": model_name}
    for cls in CLASS_NAMES:
        for m in ["precision","recall","f1-score"]:
            col = f"{cls}_{m.replace('-score','')}"
            row[col] = round(report[cls][m], 4)
    for avg in ["macro avg","weighted avg"]:
        prefix = avg.replace(" avg","")
        for m in ["precision","recall","f1-score"]:
            col = f"{prefix}_{m.replace('-score','')}"
            row[col] = round(report[avg][m], 4)
    row["accuracy"] = round(report["accuracy"], 4)
    return row

In [4]:
all_results = [] 

from sklearn.neighbors         import KNeighborsClassifier
from sklearn.tree              import DecisionTreeClassifier
from sklearn.ensemble          import RandomForestClassifier
from sklearn.naive_bayes       import GaussianNB
from sklearn.linear_model      import LogisticRegression
from sklearn.svm               import LinearSVC, SVC
from sklearn.model_selection   import GridSearchCV
from sklearn.pipeline          import Pipeline
import numpy as np

unique_classes = np.unique(y_train)
N_CLASSES = len(unique_classes)
CLASS_NAMES = [str(c) for c in unique_classes]

print("N_CLASSES :", N_CLASSES)
print("CLASS_NAMES:", CLASS_NAMES)

ML_CONFIGS = [
    (
        "KNN",
        KNeighborsClassifier(),
        {"n_neighbors":[3,5,7,11], "weights":["uniform","distance"],
         "metric":["euclidean","manhattan"]}
    ),
    (
        "DecisionTree",
        DecisionTreeClassifier(random_state=42),
        {"max_depth":[None,10,20,30], "min_samples_split":[2,5,10],
         "criterion":["gini","entropy"]}
    ),
    (
        "RandomForest",
        RandomForestClassifier(random_state=42, n_jobs=-1),
        {"n_estimators":[100,200], "max_depth":[None,10,20],
         "min_samples_split":[2,5]}
    ),
    (
        "GaussianNB",
        GaussianNB(),
        {"var_smoothing":[1e-9,1e-8,1e-7,1e-6]}
    ),
    (
        "LogisticRegression",
        LogisticRegression(max_iter=1000, random_state=42, n_jobs=-1),
        {"C":[0.01,0.1,1,10], "solver":["lbfgs","saga"],
         "multi_class":["multinomial"]}
    ),
    (
        "LinearSVM",
        LinearSVC(max_iter=2000, random_state=42),
        {"C":[0.01,0.1,1,10]}
    ),
    (
        "RBF_SVM",
        SVC(kernel="rbf", random_state=42),
        {"C":[0.1,1,10], "gamma":["scale","auto"]}
    ),
]

import pandas as pd
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import classification_report

all_results = []
report_rows = []

for name, estimator, param_grid in ML_CONFIGS:
    print(f"\n[{name}] GridSearch đang chạy...")

    gs = GridSearchCV(
        estimator,
        param_grid,
        cv=5,
        scoring="f1_macro",
        n_jobs=-1,
        verbose=0
    )

    gs.fit(X_train_selected, y_train)

    best = gs.best_estimator_
    y_pred = best.predict(X_test_selected)

    print(f"  Best params : {gs.best_params_}")
    print(f"  CV F1-macro : {gs.best_score_:.4f}")

    # report dạng dict thay vì in text thô
    report_dict = classification_report(
        y_test,
        y_pred,
        target_names=CLASS_NAMES,
        zero_division=0,
        output_dict=True
    )

    # Gom toàn bộ report của model thành 1 hàng
    flat_row = {"model": name}

    for label, metrics in report_dict.items():
        if isinstance(metrics, dict):  # các class, macro avg, weighted avg
            for metric_name, value in metrics.items():
                flat_row[f"{label}_{metric_name}"] = value
        else:  # accuracy là scalar
            flat_row[label] = metrics

    flat_row["best_params"] = str(gs.best_params_)
    flat_row["cv_f1_macro"] = gs.best_score_
    report_rows.append(flat_row)

    # phần cũ của bạn nếu vẫn muốn giữ
    row = compute_metrics(y_test, y_pred, name)
    row["best_params"] = str(gs.best_params_)
    all_results.append(row)

# Bảng classification report: mỗi hàng là 1 model
classification_report_df = pd.DataFrame(report_rows)

# Sắp xếp cột cho dễ nhìn
first_cols = ["model", "accuracy", "cv_f1_macro", "best_params"]
other_cols = [c for c in classification_report_df.columns if c not in first_cols]
classification_report_df = classification_report_df[first_cols + other_cols]

display(classification_report_df)

pd.DataFrame(all_results).to_csv(
    "all_results_models.csv",
    index=False,
    encoding="utf-8-sig"
)

N_CLASSES : 8
CLASS_NAMES: ['0', '1', '2', '3', '4', '5', '6', '7']

[KNN] GridSearch đang chạy...
  Best params : {'metric': 'manhattan', 'n_neighbors': 11, 'weights': 'distance'}
  CV F1-macro : 0.5572

[DecisionTree] GridSearch đang chạy...
  Best params : {'criterion': 'entropy', 'max_depth': 10, 'min_samples_split': 10}
  CV F1-macro : 0.6138

[RandomForest] GridSearch đang chạy...
  Best params : {'max_depth': None, 'min_samples_split': 5, 'n_estimators': 200}
  CV F1-macro : 0.7587

[GaussianNB] GridSearch đang chạy...
  Best params : {'var_smoothing': 1e-06}
  CV F1-macro : 0.3607

[LogisticRegression] GridSearch đang chạy...


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and wi

  Best params : {'C': 0.01, 'multi_class': 'multinomial', 'solver': 'lbfgs'}
  CV F1-macro : 0.7517

[LinearSVM] GridSearch đang chạy...
  Best params : {'C': 0.01}
  CV F1-macro : 0.7490

[RBF_SVM] GridSearch đang chạy...
  Best params : {'C': 10, 'gamma': 'auto'}
  CV F1-macro : 0.8190


,model,accuracy,cv_f1_macro,best_params,0_precision,0_recall,0_f1-score,0_support,1_precision,1_recall,...,7_f1-score,7_support,macro avg_precision,macro avg_recall,macro avg_f1-score,macro avg_support,weighted avg_precision,weighted avg_recall,weighted avg_f1-score,weighted avg_support
0,KNN,0.5760,0.557173,"{'metric': 'manhattan', 'n_neighbors': 11, 'we...",0.588235,0.240,0.340909,250.0,0.435294,0.888,...,0.643026,250.0,0.615010,0.5760,0.567011,2000.0,0.615010,0.5760,0.567011,2000.0
1,DecisionTree,0.6265,0.613754,"{'criterion': 'entropy', 'max_depth': 10, 'min...",0.454545,0.400,0.425532,250.0,0.641566,0.852,...,0.655303,250.0,0.628216,0.6265,0.623280,2000.0,0.628216,0.6265,0.623280,2000.0
2,RandomForest,0.7660,0.758676,"{'max_depth': None, 'min_samples_split': 5, 'n...",0.765363,0.548,0.638695,250.0,0.734426,0.896,...,0.777778,250.0,0.774481,0.7660,0.763337,2000.0,0.774481,0.7660,0.763337,2000.0
3,GaussianNB,0.3740,0.360749,{'var_smoothing': 1e-06},0.133333,0.040,0.061538,250.0,0.397059,0.864,...,0.484375,250.0,0.393683,0.3740,0.354982,2000.0,0.393683,0.3740,0.354982,2000.0
4,LogisticRegression,0.7505,0.751666,"{'C': 0.01, 'multi_class': 'multinomial', 'sol...",0.680000,0.544,0.604444,250.0,0.768683,0.864,...,0.765531,250.0,0.748733,0.7505,0.748119,2000.0,0.748733,0.7505,0.748119,2000.0
5,LinearSVM,0.7455,0.749027,{'C': 0.01},0.693122,0.524,0.596811,250.0,0.780069,0.908,...,0.759443,250.0,0.742324,0.7455,0.741647,2000.0,0.742324,0.7455,0.741647,2000.0
6,RBF_SVM,0.8225,0.818993,"{'C': 10, 'gamma': 'auto'}",0.714286,0.680,0.696721,250.0,0.834586,0.888,...,0.828974,250.0,0.822562,0.8225,0.822259,2000.0,0.822562,0.8225,0.822259,2000.0
